##### Data ingestion

In [1]:
# import the libraries we need
import pandas as pd
import glob

# load files
path = './spotify_data/*.json' 
files = glob.glob(path)

# read each file into a DataFrame and store them in a list
data_frames = [pd.read_json(f) for f in files]

# combine all the DataFrames into one big DataFrame
full_history = pd.concat(data_frames, ignore_index=True)

print(full_history.head())

            endTime         artistName  \
0  2024-11-30 14:05               RAYE   
1  2024-12-06 21:41          Adi Oasis   
2  2024-12-09 18:20         Steve Lacy   
3  2024-12-10 13:12  María José Llergo   
4  2024-12-10 13:12           Dua Lipa   

                                trackName  msPlayed  
0                               Worth It.     22607  
1                               Le Départ     12129  
2                                    Uuuu     28256  
3  RUEDA, RUEDA - Live at NPR's Tiny Desk      3847  
4                                 Houdini      1932  


##### Feature engineering and normalizing

In [2]:
# import the library for normalization
from sklearn.preprocessing import MinMaxScaler

# select the columns we want to turn into numbers
df_for_ml = full_history[['artistName', 'msPlayed']]

# one-hot encoding (turning names into 1s and 0s)
encoded_artists = pd.get_dummies(df_for_ml['artistName'])

# normalization (shrinking the play time to 0-1)
scaler = MinMaxScaler()

# normalize the play time
ms_scaled = scaler.fit_transform(df_for_ml[['msPlayed']])

# create a DataFrame with the scaled values
ms_scaled_df = pd.DataFrame(ms_scaled, columns=['msPlayed_scaled'], index=df_for_ml.index)

# combine the encoded artists and the scaled play time into one DataFrame
ml_matrix = pd.concat([encoded_artists, ms_scaled_df], axis=1)

# check the result
print(ml_matrix.head())

    10cc  2 Chainz  21 Savage   2Pac   32ki  3BallMTY  4 Non Blondes  50 Cent  \
0  False     False      False  False  False     False          False    False   
1  False     False      False  False  False     False          False    False   
2  False     False      False  False  False     False          False    False   
3  False     False      False  False  False     False          False    False   
4  False     False      False  False  False     False          False    False   

   6ix9ine  8twenty  ...  梶浦 由記   椎乃味醂    椎名豪  稲垣次郎とソウル・メディア     美波  \
0    False    False  ...  False  False  False          False  False   
1    False    False  ...  False  False  False          False  False   
2    False    False  ...  False  False  False          False  False   
3    False    False  ...  False  False  False          False  False   
4    False    False  ...  False  False  False          False  False   

   音羽-otoha-   須田景凪  高橋　洋子  ꉈꀧ꒒꒒ꁄꍈꍈꀧ꒦ꉈ ꉣꅔꎡꅔꁕꁄ  msPlayed_scaled  
0      False  False  

##### Mathematical modeling stage

In [3]:
# group by artist for memory efficiency (2,350 rows instead of 30,000)
artist_matrix = ml_matrix.groupby(full_history['artistName']).mean()

# run the similarity on the smaller matrix
from sklearn.metrics.pairwise import cosine_similarity
artist_sim = cosine_similarity(artist_matrix)

# create the DataFrame
artist_sim_df = pd.DataFrame(artist_sim, index=artist_matrix.index, columns=artist_matrix.index)

# sorting and testing: What artists are similar to RAYE?
print(artist_sim_df['RAYE'].sort_values(ascending=False).head(10))

artistName
RAYE                  1.000000
mark william lewis    0.022879
Céline Dion           0.019440
Líricas del Caos      0.018792
Machabara             0.018671
Sampa the Great       0.018254
Miles Caton           0.017832
Bassolino             0.017449
Vangelis              0.017068
Johann Debussy        0.016900
Name: RAYE, dtype: float64


##### Data enrichment and security stage

In [4]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# load the keys from the .env file
load_dotenv()

# grab the keys from the "environment"
client_id = os.getenv('SPOTIPY_CLIENT_ID')
client_secret = os.getenv('SPOTIPY_CLIENT_SECRET')

# connect to Spotify
auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager)

# create an empty list to hold the artist data
artist_data = []
unique_artists = full_history['artistName'].unique()

# process all artists in logical batches
print(f"Starting API retrieval for {len(unique_artists)} artists...")

# process artists in batches of 50 to make the data more manageable
for i in range(0, len(unique_artists), 50):
    batch = unique_artists[i:i+50]

    # loop through each artist in the batch
    for artist_name in batch:
        # search for the artist
        results = sp.search(q='artist:' + artist_name, type='artist', limit=1)
        items = results['artists']['items']
        
        # ensure every artist gets a row, even if genres are missing
        if items:
            genres = items[0]['genres']
            artist_data.append({'artistName': artist_name, 'genres': genres})
        else:
            # create an 'Empty List' is vital for keeping your matrix clean later
            artist_data.append({'artistName': artist_name, 'genres': []})

# final table of artists and their genres
genre_df = pd.DataFrame(artist_data)
print("Data enrichment complete.")

Starting API retrieval for 2349 artists...
Data enrichment complete.


##### Quality control & data pruning

In [5]:
# filter out artists where the genre list is empty
genre_df_clean = genre_df[genre_df['genres'].map(len) > 0].copy()

# check how many artists are left
print(f"Original count: {len(genre_df)}")
print(f"Cleaned count: {len(genre_df_clean)}")


Original count: 2349
Cleaned count: 1466


In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

# turns the genre lists into a binary matrix of 1s and 0s
mlb = MultiLabelBinarizer()

# create the genre grid
genre_matrix = pd.DataFrame(
    mlb.fit_transform(genre_df_clean['genres']),
    columns=mlb.classes_,
    index=genre_df_clean['artistName']
)

# combine with the scaled play time
final_ml_matrix = pd.concat([genre_matrix, artist_matrix['msPlayed_scaled']], axis=1, join='inner')

print("Final Matrix Shape:", final_ml_matrix.shape)


Final Matrix Shape: (1466, 480)


##### Multidimensional similarity mapping

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# calculate similarity
final_sim = cosine_similarity(final_ml_matrix)
final_sim_df = pd.DataFrame(final_sim, index=final_ml_matrix.index, columns=final_ml_matrix.index)

# see who is similar to Adi Oasis
if 'Adi Oasis' in final_sim_df.index:
    print(final_sim_df['Adi Oasis'].sort_values(ascending=False).head(10))

artistName
Adi Oasis                          1.000000
Aaron Taylor                       0.816007
Hiatus Kaiyote                     0.666023
Ms. Lauryn Hill                    0.577384
Sampha                             0.577384
Leon Bridges                       0.577382
LEISURE                            0.577377
Thee Sacred Souls                  0.577371
Gabriels                           0.577358
The Flying Stars Of Brooklyn NY    0.577358
Name: Adi Oasis, dtype: float64


##### Discovery engine: using Spotify API as a "prospector"

In [8]:
# check how many times each genre appears
top_genres = genre_df_clean.explode('genres')['genres'].value_counts()
print(top_genres.head(20))

genres
reggaeton            93
j-pop                54
salsa                53
anime                51
rock en español      49
latin rock           41
merengue             41
latin pop            39
musicals             37
indie                36
surf rock            33
latin alternative    32
urbano latino        32
rock                 29
j-rock               26
latin                26
disco                24
trap latino          24
classic rock         24
lounge               24
Name: count, dtype: int64


In [9]:
# shows you all the genres the model is using
print(final_ml_matrix.columns.tolist())

['acid jazz', 'acid rock', 'acid techno', 'adult standards', 'afro house', 'afro r&b', 'afro soul', 'afro tech', 'afro-cuban jazz', 'afrobeat', 'afrobeats', 'afropiano', 'afropop', 'afroswing', 'alternative dance', 'alternative hip hop', 'alternative metal', 'alternative r&b', 'alternative rock', 'alté', 'ambient', 'ambient jazz', 'anatolian rock', 'anime', 'anti-folk', 'aor', 'argentine rock', 'argentine trap', 'arrocha', 'art pop', 'art rock', 'axé', 'bachata', 'ballet', 'ballroom vogue', 'banda', 'baroque', 'baroque pop', 'bass house', 'bassline', 'bedroom pop', 'bhajan', 'big band', 'big beat', 'big room', 'blues', 'blues rock', 'bolero', 'bollywood', 'boom bap', 'bossa nova', 'brazilian bass', 'brazilian funk', 'brazilian hip hop', 'brazilian jazz', 'brazilian phonk', 'brazilian pop', 'brazilian rock', 'brazilian trap', 'breakbeat', 'breakcore', 'brega funk', 'britpop', 'brooklyn drill', 'candombe', 'canzone napoletana', 'celtic rock', 'cha cha cha', 'chamamé', 'chamber music', 'c

In [10]:
# pick a genre you love (from your high similarity results)
target_genre = 'indie soul' 

# search for artists in that genre
search_results = sp.search(q=f'genre:"{target_genre}"', type='artist', limit=50)

# create an empty list to hold the new discovery artists
new_discovery_list = []

# loop through the search results and check if they are NOT in your current library of 1,525 artists
for artist in search_results['artists']['items']:
    if artist['name'] not in genre_df_clean['artistName'].values:
        new_discovery_list.append({
            'name': artist['name'],
            'genres': artist['genres'],
            'popularity': artist['popularity']
        })

# create the discovery table
discovery_df = pd.DataFrame(new_discovery_list)
print(f"Found {len(discovery_df)} new artists in the '{target_genre}' genre!")
print(discovery_df[['name', 'popularity']].head(10))

Found 37 new artists in the 'indie soul' genre!
               name  popularity
0  Ben L'Oncle Soul          53
1              JMSN          66
2            Unjaps          71
3         Moon Soul          54
4       Olive Jones          60
5       James Blake          72
6             Re:um          57
7         Baby Rose          55
8      Aaron Childs          52
9    Octave Lissner          45


##### Predictive scoring: aligning novel candidates with the "taste profile"

**Note**: Identical scores indicate that these artists share the exact same genre metadata on Spotify, making them equally strong candidates according to the current feature set.

In [11]:
# transform the new artists' genres using your existing MLB (MultiLabelBinarizer) 'exploder'
new_genre_matrix = pd.DataFrame(
    mlb.transform(discovery_df['genres']), # transform uses your EXISTING 491 columns
    columns=mlb.classes_,
    index=discovery_df['name']
)

# add a 'fake' play-time (since we haven't heard them, we'll give them a neutral 0.5)
new_genre_matrix['msPlayed_scaled'] = 0.5

# 3. Compare them to your 'Taste Profile' (the average of your favorite artists)
taste_profile = final_ml_matrix.mean(axis=0).values.reshape(1, -1)
discovery_sim = cosine_similarity(new_genre_matrix, taste_profile)

# 4. Show the winners!
discovery_df['match_score'] = discovery_sim
print(discovery_df.sort_values(by='match_score', ascending=False)[['name', 'match_score']].head(10))

              name  match_score
18          Jungle     0.247707
16      Leven Kali     0.247707
35  Free Nationals     0.247707
34    Holly Walker     0.247707
33         Redinho     0.247707
30   Maribou State     0.247707
29           RUBII     0.247707
28       m. demian     0.247707
25     Benny Sings     0.247707
24      WAYNE SNOW     0.247707


In [12]:
# get the average of ONLY top 10 artists
top_10_artists = final_ml_matrix.sort_values('msPlayed_scaled', ascending=False).head(10)
target_taste_vector = top_10_artists.mean(axis=0).values.reshape(1, -1)

# re-calculate the match score for the new artists
discovery_sim_refined = cosine_similarity(new_genre_matrix, target_taste_vector)
discovery_df['refined_match'] = discovery_sim_refined

print(discovery_df.sort_values(by='refined_match', ascending=False)[['name', 'refined_match']].head(5))

              name  refined_match
18          Jungle       0.472745
16      Leven Kali       0.472745
35  Free Nationals       0.472745
34    Holly Walker       0.472745
33         Redinho       0.472745
